============================================================================
# Implementación de agentización utilizando un pipeline de dominios específicos con LangGraph
============================================================================

In [ ]:
import os
import json
from pathlib import Path
from typing import Dict, List, Any, Optional
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
import numpy as np

# Cargar variables de entorno
load_dotenv()

In [ ]:
azure_model = AzureChatOpenAI(
    azure_deployment="csbridge-gpt-4o-mini",
    azure_endpoint="https://csbridgeopenai.openai.azure.com/",
    api_version="2024-02-15-preview",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    temperature=0.3,
    max_tokens=4096
)

============================================================================
## ESQUEMAS DE ESTADO
============================================================================

In [ ]:
class DomainAwareLegalState(BaseModel):
    """Estado del caso legal con clasificación de dominio"""
    case_id: str = Field(description="ID del caso")
    original_text: str = Field(description="Texto original del juicio")
    predicted_area_of_law: Optional[str] = Field(default=None, description="Área de derecho predicha")
    domain_characteristics: Optional[Dict[str, Any]] = Field(default=None, description="Características del dominio")
    extracted_fields: Optional[Dict[str, Any]] = Field(default=None, description="Campos extraídos")
    summary: Optional[str] = Field(default=None, description="Resumen generado")
    error_message: Optional[str] = Field(default=None, description="Mensaje de error si falla")
    processing_step: str = Field(default="start", description="Paso actual del procesamiento")
    retry_count: int = Field(default=0, description="Número de reintentos")
    max_retries: int = Field(default=2, description="Máximo de reintentos")

============================================================================
## CARGA DE CARACTERÍSTICAS DE DOMINIO
===========================================================================

In [ ]:
def load_domain_characteristics():
    """Carga las características de dominios legales desde ids_by_legal_domain.json"""
    try:
        with open('resultados/ids_by_legal_domain.json', 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        domain_characteristics = {}
        for group in data.get('legal_domain_groups', []):
            area_of_law = group['area_of_law']
            section_analysis = group.get('section_analysis', {})
            
            domain_characteristics[area_of_law] = {
                'most_common_order': section_analysis.get('most_common_order', []),
                'order_confidence': section_analysis.get('order_confidence', 0),
                'section_length_analysis': section_analysis.get('section_length_analysis', {})
            }
        
        print(f"Loaded characteristics for {len(domain_characteristics)} legal domains")
        return domain_characteristics
        
    except FileNotFoundError:
        print("❌ Domain characteristics file not found. Using default characteristics.")
        return {}


============================================================================
## PROMPTS ESPECÍFICOS POR DOMINIO
============================================================================

In [ ]:
DOMAIN_CLASSIFICATION_PROMPT = """You are an expert legal domain classifier for Indian judgments.

Your task: Classify the following legal judgment into ONE of these specific areas of law:

{legal_domains}

CRITICAL INSTRUCTIONS:
- Read the judgment carefully and identify the primary legal domain
- Consider the subject matter, legal issues, and procedural context
- Return ONLY the exact area of law from the list above
- If uncertain, choose the most appropriate domain based on the main legal issues

LEGAL JUDGMENT TEXT:
{text}

CLASSIFICATION:"""

def create_domain_specific_extraction_prompt(domain_characteristics: Dict[str, Any], text: str) -> str:
    """Crea un prompt de extracción específico para el dominio"""
    
    most_common_order = domain_characteristics.get('most_common_order', [])
    section_length_analysis = domain_characteristics.get('section_length_analysis', {})
    
    # Construir información de secciones esperadas
    sections_info = ""
    for section in most_common_order:
        length_info = section_length_analysis.get(section, {})
        avg_words = length_info.get('avg_words', 0)
        avg_words_per_sentence = length_info.get('avg_words_per_sentence', 0)
        
        sections_info += f"\n- {section}: ~{avg_words} words, ~{avg_words_per_sentence} words/sentence"
    
    return f"""You are an expert legal information extractor for Indian judgments in the {domain_characteristics.get('domain', 'legal')} domain.

Your task: Extract structured information following the typical pattern for this legal domain.

EXPECTED SECTION ORDER FOR THIS DOMAIN:
{most_common_order}

TARGET LENGTHS FOR EACH SECTION:
{sections_info}

CRITICAL INSTRUCTIONS FOR MAXIMUM ROUGE-L AND BLEU:
- COPY verbatim phrases and sentences from the original text
- DO NOT paraphrase or rephrase — extract exact textual segments
- Preserve original wording, terminology, and sentence structure
- Use literal quotes from the judgment for all fields
- Maintain exact punctuation, capitalization, and legal terminology
- Follow the expected section order for this domain

Required JSON schema (exact keys):
{{
  "ID": "<use provided id when available or empty>",
  "Court": "", 
  "Date": "", 
  "Parties": {{ "Petitioner": [], "Respondent": [] }},
  "Counsel": [],              # list of counsel names (verbatim from text)
  "ProceduralHistory": "",    # copy literal sentence(s) describing procedural background
  "Facts": [],                # array of verbatim sentences describing key facts
  "Issues": [],               # array of verbatim phrases/sentences stating legal questions
  "Arguments": [],            # array of verbatim sentences describing arguments
  "Decision": "",             # copy exact outcome statement from judgment
  "Reasoning": [],            # array of verbatim sentences explaining court's reasoning
  "Orders": "",               # copy exact operative orders/reliefs text
  "Citations": []             # list of exact case names / statutes as mentioned
}}

IMPORTANT:
- Output MUST be valid JSON only
- Extract by COPYING literal text segments — do not summarize or paraphrase
- If a field is not present, use empty string or []
- Preserve exact names, dates, legal terms from the original judgment
- Follow the domain-specific section order and length guidelines

LEGAL JUDGMENT TEXT:
{text}"""

def create_domain_specific_abstraction_prompt(domain_characteristics: Dict[str, Any], extracted_fields: Dict[str, Any]) -> str:
    """Crea un prompt de abstracción específico para el dominio"""
    
    most_common_order = domain_characteristics.get('most_common_order', [])
    section_length_analysis = domain_characteristics.get('section_length_analysis', {})
    
    # Construir guías de longitud por sección
    length_guidelines = ""
    for section in most_common_order:
        length_info = section_length_analysis.get(section, {})
        avg_words = length_info.get('avg_words', 0)
        if avg_words > 0:
            length_guidelines += f"\n- {section}: Target ~{avg_words} words"
    
    return f"""You are an expert legal summarizer optimized for MAXIMUM ROUGE-L and BLEU scores for {domain_characteristics.get('domain', 'legal')} domain cases.

CRITICAL OPTIMIZATION STRATEGY:
- REUSE verbatim phrases and sentences from the extracted fields
- COPY literal n-grams (3-5+ word sequences) from the source text
- MINIMIZE paraphrasing — preserve original wording wherever possible
- MAINTAIN exact legal terminology, names, dates, and citations
- Build summary by CONCATENATING and ARRANGING literal text segments
- FOLLOW the domain-specific section order and length guidelines
- DO NOT use section headers or titles - write as a flowing narrative

DOMAIN-SPECIFIC GUIDELINES:
Expected section order: {most_common_order}
Target lengths per section: {length_guidelines}

Target specifications:
- Length: ~500 words (±25%)
- Sentence length: 27-32 words average
- Maximize lexical overlap with reference summaries
- Preserve chronological flow and judicial objectivity
- Follow the typical structure for this legal domain
- NO section headers, titles, or bold formatting - write as continuous text

ASSEMBLY INSTRUCTIONS:
1. Start with verbatim party names and court/date information
2. Follow the domain-specific section order: {most_common_order}
3. Incorporate literal sentences from Facts, Issues, Arguments
4. Copy exact Decision and Reasoning statements
5. Include verbatim Orders and Citations
6. Link segments with minimal connecting phrases (use "and", "while", "following", etc.)
7. DO NOT create new phrasings — recombine existing text
8. Respect the target lengths for each section type
9. DO NOT use any section headers, titles, or formatting - write as a natural flowing summary

Input: Use the extracted fields below and REUSE their literal text.

{extracted_fields}

Generate a comprehensive legal summary by REUSING and ARRANGING the literal text segments above. Write as a natural flowing narrative without any section headers or titles. Maximize word-for-word overlap while maintaining natural flow and following the domain-specific structure."""


============================================================================
## FUNCIONES DE PROCESAMIENTO (NODOS DEL GRAFO)
============================================================================

In [ ]:
def classify_legal_domain(state: DomainAwareLegalState) -> DomainAwareLegalState:
    """Nodo 1: Clasifica el área de derecho del caso"""
    print(f"🔍 Classifying legal domain for case {state.case_id}")
    
    try:
        # Cargar características de dominio
        domain_characteristics = load_domain_characteristics()
        legal_domains = list(domain_characteristics.keys())
        
        if not legal_domains:
            # Fallback si no hay características
            print("❌ No hay características de dominio disponibles, usando fallback")
            legal_domains = ["Criminal Law", "Civil Law", "Family Law", "Administrative Law", "Constitutional Law"]
        
        # Crear prompt de clasificación
        classification_prompt = DOMAIN_CLASSIFICATION_PROMPT.format(
            legal_domains="\n".join(f"- {domain}" for domain in legal_domains),
            text=state.original_text[:5000]  # Truncar para evitar context length
        )
        
        # Llamar al modelo
        response = azure_model.invoke([
            {"role": "system", "content": "You are a legal domain classifier. Return only the exact area of law name."},
            {"role": "user", "content": classification_prompt}
        ])
        
        predicted_domain = response.content.strip()
        state.predicted_area_of_law = predicted_domain
        
        # Cargar características del dominio predicho
        if predicted_domain in domain_characteristics:
            state.domain_characteristics = domain_characteristics[predicted_domain]
            state.domain_characteristics['domain'] = predicted_domain
            state.processing_step = "classification_success"
            print(f"Classified as: {predicted_domain}")
        else:
            # Fallback para dominios no reconocidos
            state.domain_characteristics = {
                'most_common_order': ["Case Information", "Facts", "Issues", "Arguments", "Decision", "Reasoning"],
                'order_confidence': 0,
                'section_length_analysis': {},
                'domain': predicted_domain
            }
            state.processing_step = "classification_fallback"
            print(f"Unknown domain '{predicted_domain}', using fallback characteristics")
            
    except Exception as e:
        state.error_message = f"Classification failed: {str(e)}"
        state.processing_step = "classification_failed"
        print(f"❌ Classification failed for case {state.case_id}: {e}")
    
    return state

def extract_structured_fields(state: DomainAwareLegalState) -> DomainAwareLegalState:
    """Nodo 2: Extrae campos estructurados usando características del dominio"""
    print(f"🔍 Extracting structured fields for case {state.case_id} (domain: {state.predicted_area_of_law})")
    
    try:
        if not state.domain_characteristics:
            state.error_message = "No domain characteristics available for extraction"
            state.processing_step = "extraction_failed"
            return state
        
        # Truncar texto si es muy largo
        text = state.original_text
        if len(text) > 100000:
            text = text[:100000] + "... [truncated]"
            print(f"Text truncated for case {state.case_id}")
        
        # Crear prompt específico del dominio
        extraction_prompt = create_domain_specific_extraction_prompt(
            state.domain_characteristics, 
            text
        )
        
        # Llamar al modelo
        response = azure_model.invoke([
            {"role": "system", "content": "You are a structured data extractor for legal judgments. Return only JSON or a JSON-like block."},
            {"role": "user", "content": extraction_prompt}
        ])
        
        # Parsear respuesta como JSON
        extracted_content = response.content.strip()
        
        try:
            extracted_fields = json.loads(extracted_content)
            state.extracted_fields = extracted_fields
            state.processing_step = "extraction_success"
            print(f"Extraction successful for case {state.case_id}")
        except json.JSONDecodeError:
            state.extracted_fields = {"raw_extraction": extracted_content}
            state.processing_step = "extraction_partial"
            print(f"Extraction returned non-JSON for case {state.case_id}")
            
    except Exception as e:
        state.error_message = f"Extraction failed: {str(e)}"
        state.processing_step = "extraction_failed"
        print(f"❌ Extraction failed for case {state.case_id}: {e}")
    
    return state

def generate_abstract_summary(state: DomainAwareLegalState) -> DomainAwareLegalState:
    """Nodo 3: Genera resumen abstractivo usando características del dominio"""
    print(f"📝 Generating domain-aware abstract summary for case {state.case_id}")
    
    try:
        if not state.extracted_fields or not state.domain_characteristics:
            state.error_message = "No extracted fields or domain characteristics available"
            state.processing_step = "abstraction_failed"
            return state
        
        # Convertir campos extraídos a string
        if isinstance(state.extracted_fields, dict):
            extracted_str = json.dumps(state.extracted_fields, ensure_ascii=False, indent=2)
        else:
            extracted_str = str(state.extracted_fields)
        
        # Crear prompt específico del dominio
        abstraction_prompt = create_domain_specific_abstraction_prompt(
            state.domain_characteristics,
            extracted_str
        )
        
        # Llamar al modelo
        response = azure_model.invoke([
            {"role": "system", "content": f"You are an expert legal document summarizer, specialized in {state.predicted_area_of_law}."},
            {"role": "user", "content": abstraction_prompt}
        ])
        
        state.summary = response.content.strip()
        state.processing_step = "abstraction_success"
        print(f"Domain-aware abstraction successful for case {state.case_id} ({len(state.summary.split())} words)")
        
    except Exception as e:
        state.error_message = f"Abstraction failed: {str(e)}"
        state.processing_step = "abstraction_failed"
        print(f"❌ Abstraction failed for case {state.case_id}: {e}")
    
    return state

def handle_failure(state: DomainAwareLegalState) -> DomainAwareLegalState:
    """Nodo 4: Maneja fallos y aplica resumen genérico"""
    print(f"🔄 Handling failure for case {state.case_id}")
    
    # Resumen genérico de fallback
    GENERIC_SUMMARY = "This legal case involves judicial proceedings where the court examined the matter presented by the parties. The judgment addresses the legal arguments and evidence submitted during the hearing and provides a resolution based on applicable law."
    
    state.summary = GENERIC_SUMMARY
    state.processing_step = "fallback_applied"
    print(f"Applied fallback summary for case {state.case_id}")
    
    return state

def should_retry(state: DomainAwareLegalState) -> str:
    """Función de decisión: ¿debe reintentar o continuar?"""
    if state.retry_count < state.max_retries and state.processing_step in ["classification_failed", "extraction_failed", "abstraction_failed"]:
        state.retry_count += 1
        print(f"🔄 Retrying case {state.case_id} (attempt {state.retry_count}/{state.max_retries})")
        return "retry"
    elif state.processing_step == "abstraction_success":
        return "success"
    else:
        return "failure"


============================================================================
## CONSTRUCCIÓN DEL GRAFO LANGGRAPH
============================================================================

In [ ]:
def create_domain_aware_pipeline_graph():
    """Crea el grafo de LangGraph para el pipeline con clasificación de dominio"""
    
    # Crear el grafo
    workflow = StateGraph(DomainAwareLegalState)
    
    # Agregar nodos
    workflow.add_node("classify", classify_legal_domain)
    workflow.add_node("extract", extract_structured_fields)
    workflow.add_node("abstract", generate_abstract_summary)
    workflow.add_node("handle_failure", handle_failure)
    workflow.add_node("success", lambda state: state)
    
    # Definir flujo
    workflow.set_entry_point("classify")
    
    # Desde classify: ir a extract si éxito, a handle_failure si falla
    workflow.add_conditional_edges(
        "classify",
        lambda state: "extract" if state.processing_step in ["classification_success", "classification_fallback"] else "handle_failure"
    )
    
    # Desde extract: ir a abstract si éxito, a handle_failure si falla
    workflow.add_conditional_edges(
        "extract",
        lambda state: "abstract" if state.processing_step in ["extraction_success", "extraction_partial"] else "handle_failure"
    )
    
    # Desde abstract: ir a success si éxito, a handle_failure si falla
    workflow.add_conditional_edges(
        "abstract",
        lambda state: "success" if state.processing_step == "abstraction_success" else "handle_failure"
    )
    
    # Conexiones explícitas
    workflow.add_edge("success", END)
    workflow.add_edge("handle_failure", END)
    
    # Compilar el grafo con checkpointing
    app = workflow.compile(checkpointer=InMemorySaver())
    
    return app


============================================================================
## EVALUACIÓN ROUGE/BLEU
============================================================================

In [ ]:
def evaluate_summary(generated_summary: str, reference_summary: str) -> Dict[str, float]:
    """Evalúa un resumen generado contra uno de referencia usando ROUGE y BLEU"""
    try:
        from rouge_score import rouge_scorer
        from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
        import numpy as np
        
        results = {}
        
        # ROUGE Scores
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        rouge_scores = scorer.score(reference_summary, generated_summary)
        
        results['rouge1_f'] = rouge_scores['rouge1'].fmeasure
        results['rouge2_f'] = rouge_scores['rouge2'].fmeasure
        results['rougeL_f'] = rouge_scores['rougeL'].fmeasure
        
        # BLEU Score
        try:
            reference_tokens = reference_summary.lower().split()
            generated_tokens = generated_summary.lower().split()
            smoothing = SmoothingFunction().method1
            bleu_score = sentence_bleu([reference_tokens], generated_tokens, smoothing_function=smoothing)
            results['bleu'] = bleu_score
        except Exception:
            results['bleu'] = 0.0
        
        return results
        
    except ImportError as e:
        print(f"Evaluation libraries not available: {e}")
        return {'rouge1_f': 0.0, 'rouge2_f': 0.0, 'rougeL_f': 0.0, 'bleu': 0.0}


============================================================================
## FUNCIONES DE UTILIDAD
============================================================================

In [ ]:
def process_single_case(case_data: Dict[str, Any], app) -> Dict[str, Any]:
    """Procesa un caso individual usando el grafo de LangGraph"""
    
    # Crear estado inicial
    initial_state = DomainAwareLegalState(
        case_id=case_data.get("ID", "unknown"),
        original_text=case_data.get("Judgment", ""),
        processing_step="start"
    )
    
    # Configuración para el checkpointer
    config = {
        "configurable": {
            "thread_id": f"case_{case_data.get('ID', 'unknown')}"
        }
    }
    
    # Ejecutar el grafo
    try:
        result = app.invoke(initial_state, config=config)
        
        return {
            "ID": result.get("case_id", "unknown"),
            "Summary": result.get("summary") or "Failed to generate summary",
            "PredictedAreaOfLaw": result.get("predicted_area_of_law"),
            "ProcessingStep": result.get("processing_step", "unknown"),
            "Error": result.get("error_message"),
            "RetryCount": result.get("retry_count", 0)
        }
        
    except Exception as e:
        return {
            "ID": case_data.get("ID", "unknown"),
            "Summary": "Failed to generate summary",
            "PredictedAreaOfLaw": "unknown",
            "ProcessingStep": "error",
            "Error": str(e),
            "RetryCount": 0
        }

def load_legal_data(judgments_path: str) -> List[Dict[str, Any]]:
    """Carga datos de juicios desde archivo JSONL"""
    judgments = []
    with open(judgments_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                judgments.append(json.loads(line.strip()))
    return judgments


============================================================================
## EJECUCIÓN PRINCIPAL
============================================================================

In [ ]:
# Ejecutar pipeline sobre primeras 5 filas del dataset de entrenamiento
train_path = '../datasets/train/train_judg.jsonl'
train_ref_path = '../datasets/train/train_ref_summ.jsonl'
output_path = 'resultados/answer_domain_aware_test.jsonl'

# Cargar solo 5 casos para prueba
print("Loading 5 test cases from training data...")
all_cases = load_legal_data(train_path)
test_cases = all_cases[:2]  # Solo los primeros 5

# Cargar resúmenes de referencia para evaluación
print("Loading reference summaries for evaluation...")
try:
    ref_summaries = load_legal_data(train_ref_path)
    ref_dict = {ref["ID"]: ref["Summary"] for ref in ref_summaries}
    print(f"Loaded {len(ref_dict)} reference summaries")
except Exception as e:
    print(f"Could not load reference summaries: {e}")
    ref_dict = {}

print(f"Testing with {len(test_cases)} cases")

# Crear el grafo
app = create_domain_aware_pipeline_graph()
print("Domain-aware graph created successfully")

# Procesar casos de prueba
results = []
failed_cases = []
evaluation_results = []

for i, case in enumerate(test_cases):
    print(f"\nProcessing case {i+1}/{len(test_cases)}: {case.get('ID', 'unknown')}")
    
    result = process_single_case(case, app)
    results.append(result)
    
    if result["ProcessingStep"] in ["error", "fallback_applied"]:
        failed_cases.append(result["ID"])
    
    print(f"   Status: {result['ProcessingStep']}")
    print(f"   Predicted Domain: {result.get('PredictedAreaOfLaw', 'unknown')}")
    if result["Error"]:
        print(f"   Error: {result['Error']}")
    
    # Evaluación ROUGE/BLEU si hay referencia
    case_id = case.get('ID', 'unknown')
    if case_id in ref_dict and result["ProcessingStep"] == "abstraction_success":
        print(f"   Evaluating against reference...")
        eval_result = evaluate_summary(result["Summary"], ref_dict[case_id])
        evaluation_results.append({
            "ID": case_id,
            "PredictedDomain": result.get('PredictedAreaOfLaw', 'unknown'),
            "rouge1_f": eval_result["rouge1_f"],
            "rouge2_f": eval_result["rouge2_f"],
            "rougeL_f": eval_result["rougeL_f"],
            "bleu": eval_result["bleu"]
        })
        print(f"   ROUGE-L: {eval_result['rougeL_f']:.3f}, BLEU: {eval_result['bleu']:.3f}")

# Calcular métricas promedio
if evaluation_results:
    print(f"\nEVALUATION RESULTS:")
    print(f"   • Cases evaluated: {len(evaluation_results)}")
    
    avg_rouge1 = np.mean([r["rouge1_f"] for r in evaluation_results])
    avg_rouge2 = np.mean([r["rouge2_f"] for r in evaluation_results])
    avg_rougeL = np.mean([r["rougeL_f"] for r in evaluation_results])
    avg_bleu = np.mean([r["bleu"] for r in evaluation_results])
    
    print(f"   • ROUGE-1: {avg_rouge1:.3f}")
    print(f"   • ROUGE-2: {avg_rouge2:.3f}")
    print(f"   • ROUGE-L: {avg_rougeL:.3f}")
    print(f"   • BLEU: {avg_bleu:.3f}")
    
    # Guardar resultados de evaluación
    eval_path = output_path.replace('.jsonl', '_evaluation.jsonl')
    with open(eval_path, "w", encoding="utf-8") as f:
        for eval_result in evaluation_results:
            json.dump(eval_result, f, ensure_ascii=False)
            f.write("\n")
    print(f"   • Evaluation results saved to: {eval_path}")

# Guardar resultados
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    for result in results:
        json.dump({"ID": result["ID"], "Summary": result["Summary"]}, f, ensure_ascii=False)
        f.write("\n")

print(f"\nResults saved to {output_path}")
print(f"Summary: {len(results)} total, {len(failed_cases)} failed")

print("\n🎉 Domain-Aware LangGraph Legal Pipeline completed!")
print(f"   • Output: {output_path}")
print(f"   • Total cases: {len(results)}")
print(f"   • Failed cases: {len(failed_cases)}")
if failed_cases:
    print(f"   • Failed IDs: {failed_cases[:5]}{'...' if len(failed_cases) > 5 else ''}")

# Mostrar dominios predichos
print(f"\nPREDICTED LEGAL DOMAINS:")
domain_counts = {}
for result in results:
    domain = result.get('PredictedAreaOfLaw', 'unknown')
    domain_counts[domain] = domain_counts.get(domain, 0) + 1

for domain, count in sorted(domain_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"   • {domain}: {count} cases")

if evaluation_results:
    print(f"\nPERFORMANCE METRICS:")
    print(f"   • Average ROUGE-L: {avg_rougeL:.3f}")
    print(f"   • Average BLEU: {avg_bleu:.3f}")
    print(f"   • Domain-aware features: Classification, Domain-specific prompts, Section order optimization")